# 09 — Faithfulness, completeness e minimality

Reproduzimos as três definições da seção 4 de arXiv:2211.00593: $F(M)$ e $F(C)$ usam IO−S em `END`; completeness remove o mesmo conjunto $K$ de ambos; minimality mede o efeito incremental de cada node nos contextos históricos de `minimality.py`. Usamos N=100, como a validação histórica, e estatística pareada nos mesmos prompts.

Escopo da comparação: mesmas fórmulas e head-position ablation, mas **independent balanced pABC reference** em vez da referência histórica construída no mesmo batch. O valor F(M)=3.56 do paper é um **historical target from 100,000 prompts**, usado somente como comparação externa, nunca como baseline alternativo.

Resultado de minimality desta execução: **26/26 >1% pontual, mas 24/26 com CI95** inteiramente acima de 1%. A distinção entre estimativa pontual e suporte intervalar é essencial; não tratamos os dois nodes restantes como confirmação robusta do limiar.

In [1]:
import importlib.metadata
import json
import os
import subprocess
from datetime import datetime, timezone
from pathlib import Path

import plotly.express as px
import plotly.graph_objects as go
import torch

from ioi_replication.paper import (
    PAPER_FAITHFULNESS_GAP,
    PAPER_MODEL_PERFORMANCE,
    PAPER_RETAINED_FRACTION,
    MeanAblationEvaluator,
    build_balanced_paired_dataset,
    build_paired_dataset,
    cache_grouped_mean_activations,
    evaluate_all_minimality,
    evaluate_completeness_study,
    evaluate_faithfulness,
)
from ioi_replication.runtime import load_gpt2_small, select_device

SEED_TARGET = 409
SEED_REFERENCE = 410
SEED_VALIDATION = 411
N_TARGET = 100
EXAMPLES_PER_REFERENCE_GROUP = 10
N_REFERENCE = 30 * EXAMPLES_PER_REFERENCE_GROUP
BATCH_SIZE_REFERENCE = 16
BATCH_SIZE_TARGET = 50
RANDOM_SUBSETS = 10
GREEDY_RUNS = 10
GREEDY_STEPS = 10
GREEDY_CANDIDATES = 10
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULT_PATH = PROJECT_ROOT / "results/paper/09_faithfulness_completeness_minimality.json"

device = select_device()
model = load_gpt2_small(device)
target = build_paired_dataset(model.tokenizer, n=N_TARGET, seed=SEED_TARGET)
reference = build_balanced_paired_dataset(
    model.tokenizer,
    examples_per_group=EXAMPLES_PER_REFERENCE_GROUP,
    seed=SEED_REFERENCE,
)
means = cache_grouped_mean_activations(
    model,
    reference.abc_tokens.to(device),
    reference.groups,
    positions=reference.positions,
    activation="result",
    batch_size=BATCH_SIZE_REFERENCE,
    min_group_size=EXAMPLES_PER_REFERENCE_GROUP,
    reference_config={
        "kind": "balanced_pABC",
        "seed": SEED_REFERENCE,
        "examples_per_group": EXAMPLES_PER_REFERENCE_GROUP,
    },
    reference_sha256=reference.dataset_sha256,
)
evaluator = MeanAblationEvaluator(
    model=model,
    target_tokens=target.ioi_tokens.to(device),
    positions=target.positions,
    groups=target.groups,
    group_means=means,
    io_token_ids=target.io_token_ids,
    s_token_ids=target.s_token_ids,
    batch_size=BATCH_SIZE_TARGET,
)

faithfulness = evaluate_faithfulness(evaluator)
completeness = evaluate_completeness_study(
    evaluator,
    random_count=RANDOM_SUBSETS,
    greedy_runs=GREEDY_RUNS,
    greedy_steps=GREEDY_STEPS,
    greedy_candidate_limit=GREEDY_CANDIDATES,
    seed=SEED_VALIDATION,
)
minimality = evaluate_all_minimality(evaluator)

print(
    f"device={device}; N_target={len(target)}; N_ref={len(reference)}; "
    f"min_grupo={min(means.group_counts.values())}"
)
print(
    f"F(M)={faithfulness.model.mean:+.4f}±{faithfulness.model.standard_error:.4f}; "
    f"F(C)={faithfulness.circuit.mean:+.4f}±{faithfulness.circuit.standard_error:.4f}; "
    f"gap={faithfulness.absolute_gap:.4f}±{faithfulness.absolute_gap_standard_error:.4f}; "
    f"retido={faithfulness.retained_fraction:.1%}"
)
print(
    f"greedy runs={len(completeness.greedy)}; "
    f"top5 max={completeness.top_greedy[0].result.incompleteness:.4f}; "
    f"min minimality={min(item.normalized_effect for item in minimality):.2%}; "
    f"CI>1%={sum(item.ci_supports_threshold for item in minimality)}/26"
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

device=cuda; N_target=100; N_ref=300; min_grupo=10
F(M)=+3.7325±0.1559; F(C)=+3.1198±0.1474; gap=0.6127±0.1352; retido=83.6%
greedy runs=10; top5 max=4.1216; min minimality=1.40%; CI>1%=24/26


In [2]:
def node_name(node):
    return f"{node.layer}.{node.head}@{node.position.value}"


def estimate_dict(estimate):
    return {
        "mean": estimate.mean,
        "standard_error": estimate.standard_error,
        "n": estimate.n,
    }


def ci_dict(interval):
    return [interval.low, interval.high]


def completeness_dict(result, **extra):
    return {
        **extra,
        "removed": [node_name(node) for node in result.removed],
        "model_without": estimate_dict(result.model_without),
        "circuit_without": estimate_dict(result.circuit_without),
        "incompleteness": result.incompleteness,
        "incompleteness_standard_error": result.incompleteness_standard_error,
        "incompleteness_ci95": ci_dict(result.incompleteness_ci95),
    }


source_paths = {
    "validation": "src/ioi_replication/paper/validation.py",
    "ablation": "src/ioi_replication/paper/ablation.py",
    "circuit": "src/ioi_replication/paper/circuit.py",
    "runtime": "src/ioi_replication/runtime.py",
    "notebook": "notebooks/09_faithfulness_completeness_minimality.ipynb",
}
source_commit = os.environ.get("IOI_SOURCE_COMMIT") or subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
git_dirty = bool(
    subprocess.check_output(
        ["git", "status", "--porcelain"], cwd=PROJECT_ROOT, text=True
    ).strip()
)


def git_blob(relative_path):
    return subprocess.check_output(
        ["git", "rev-parse", f"{source_commit}:{relative_path}"],
        cwd=PROJECT_ROOT,
        text=True,
    ).strip()


source_blobs = {
    name: git_blob(relative_path)
    for name, relative_path in source_paths.items()
}

artifact = {
    "methodology": {
        "comparison_scope": (
            "Same formulas and head-position ablation; independent balanced pABC "
            "reference instead of the historical same-batch reference."
        ),
        "paper_model_target_scope": "External target estimated from 100,000 prompts.",
        "greedy_source": (
            "Easy-Transformer/completeness.py: sample 10 candidates and accept only "
            "strict improvements; Algorithm 3 line 4 contains a pseudocode typo."
        ),
        "uncertainty": "Paired per-prompt differences with normal-approximation 95% CI.",
    },
    "config": {
        "model_name": "gpt2",
        "device": device.type,
        "gpu": torch.cuda.get_device_name(device) if device.type == "cuda" else None,
        "dtype": str(next(model.parameters()).dtype).replace("torch.", ""),
        "n_target": len(target),
        "n_reference": len(reference),
        "reference_min_group_size": min(means.group_counts.values()),
        "batch_size_reference": BATCH_SIZE_REFERENCE,
        "batch_size_target": BATCH_SIZE_TARGET,
        "seed_target": SEED_TARGET,
        "seed_reference": SEED_REFERENCE,
        "seed_validation": SEED_VALIDATION,
        "random_subsets": RANDOM_SUBSETS,
        "greedy_runs": GREEDY_RUNS,
        "greedy_steps": GREEDY_STEPS,
        "greedy_candidate_limit": GREEDY_CANDIDATES,
        "target_dataset_sha256": target.dataset_sha256,
        "reference_dataset_sha256": reference.dataset_sha256,
    },
    "reference": {
        "kind": "balanced_pABC",
        "group_counts": {
            f"{pattern}/{template_id}": count
            for (pattern, template_id), count in sorted(means.group_counts.items())
        },
        "semantic_alignment": True,
    },
    "faithfulness": {
        "model": estimate_dict(faithfulness.model),
        "circuit": estimate_dict(faithfulness.circuit),
        "absolute_gap": faithfulness.absolute_gap,
        "absolute_gap_standard_error": faithfulness.absolute_gap_standard_error,
        "absolute_gap_ci95": ci_dict(faithfulness.absolute_gap_ci95),
        "retained_fraction": faithfulness.retained_fraction,
        "paper_targets": {
            "model_performance": PAPER_MODEL_PERFORMANCE,
            "absolute_gap": PAPER_FAITHFULNESS_GAP,
            "retained_fraction": PAPER_RETAINED_FRACTION,
        },
        "comparison_to_paper": {
            "model_delta": faithfulness.model.mean - PAPER_MODEL_PERFORMANCE,
            "gap_delta": faithfulness.absolute_gap - PAPER_FAITHFULNESS_GAP,
            "retained_fraction_delta": faithfulness.retained_fraction - PAPER_RETAINED_FRACTION,
            "paper_target_is_external_100k_prompt_estimate": True,
        },
    },
    "completeness": {
        "empty": completeness_dict(completeness.empty, kind="empty"),
        "historical_classes": [
            completeness_dict(
                item.result,
                kind="historical_class",
                class_name=item.class_name,
            )
            for item in completeness.by_class
        ],
        "random": [
            completeness_dict(item, kind="random", sample=index)
            for index, item in enumerate(completeness.random)
        ],
        "greedy": [
            {
                "seed": run.seed,
                "initial": completeness_dict(run.initial, kind="greedy_initial"),
                "curve": [
                    completeness_dict(
                        step.result,
                        kind="greedy",
                        step=index + 1,
                        selected=node_name(step.selected),
                        candidates=[node_name(node) for node in step.candidates],
                        accepted=step.accepted,
                        candidate_incompleteness=step.candidate_result.incompleteness,
                    )
                    for index, step in enumerate(run.steps)
                ],
            }
            for run in completeness.greedy
        ],
        "greedy_top5": [
            completeness_dict(
                item.result,
                kind="greedy_top5",
                rank=rank,
                run_index=item.run_index,
                step_index=item.step_index,
                accepted=True,
            )
            for rank, item in enumerate(completeness.top_greedy, start=1)
        ],
    },
    "minimality": [
        {
            "node": node_name(item.node),
            "context": [node_name(node) for node in item.context],
            "without_node": estimate_dict(item.without_node),
            "with_node": estimate_dict(item.with_node),
            "raw_effect": item.raw_effect,
            "raw_effect_standard_error": item.raw_effect_standard_error,
            "raw_effect_ci95": ci_dict(item.raw_effect_ci95),
            "normalized_effect": item.normalized_effect,
            "normalized_effect_ci95": ci_dict(item.normalized_effect_ci95),
            "threshold_fraction": item.threshold_fraction,
            "passes_threshold": item.passes_threshold,
            "ci_supports_threshold": item.ci_supports_threshold,
        }
        for item in minimality
    ],
    "traceability": {
        "source_commit": source_commit,
        "git_dirty": git_dirty,
        "hash_method": "git rev-parse <source_commit>:<path> (normalized Git blob IDs)",
        "source_blobs": source_blobs,
        "torch_version": torch.__version__,
        "transformer_lens_version": importlib.metadata.version("transformer-lens"),
        "timestamp_utc": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
        "cached_unique_evaluations": evaluator.cached_evaluations,
        "primary_sources": {
            "paper": "https://arxiv.org/abs/2211.00593",
            "completeness": "https://github.com/redwoodresearch/Easy-Transformer/blob/ea15315dd24481e9e2ac5c3ef335d82907a1dc34/easy_transformer/completeness.py",
            "minimality": "https://github.com/redwoodresearch/Easy-Transformer/blob/ea15315dd24481e9e2ac5c3ef335d82907a1dc34/easy_transformer/minimality.py",
        },
    },
}
RESULT_PATH.parent.mkdir(parents=True, exist_ok=True)
RESULT_PATH.write_text(
    json.dumps(artifact, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print(f"salvo: {RESULT_PATH}")

salvo: C:\Users\Marcos\Documents\IOI Replication\results\paper\09_faithfulness_completeness_minimality.json


In [3]:
faithfulness_fig = px.bar(
    x=["F(M)", "F(C)"],
    y=[faithfulness.model.mean, faithfulness.circuit.mean],
    error_y=[faithfulness.model.standard_error, faithfulness.circuit.standard_error],
    labels={"x": "Sistema", "y": "IO−S em END"},
    title="Faithfulness: modelo completo vs circuito publicado",
)
faithfulness_fig.add_hline(
    y=PAPER_MODEL_PERFORMANCE,
    line_dash="dash",
    annotation_text="paper F(M)=3.56",
)
faithfulness_fig.show()

completeness_rows = []
for item in completeness.by_class:
    completeness_rows.append(
        {
            "kind": "historical_class",
            "label": item.class_name,
            "F(C−K)": item.result.circuit_without.mean,
            "F(M−K)": item.result.model_without.mean,
            "incompleteness": item.result.incompleteness,
        }
    )
for index, item in enumerate(completeness.random):
    completeness_rows.append(
        {
            "kind": "random",
            "label": f"random {index + 1}",
            "F(C−K)": item.circuit_without.mean,
            "F(M−K)": item.model_without.mean,
            "incompleteness": item.incompleteness,
        }
    )
greedy_rows = []
for run_index, run in enumerate(completeness.greedy):
    for step_index, step in enumerate(run.steps, start=1):
        row = {
            "run": f"run {run_index + 1}",
            "step": step_index,
            "accepted": step.accepted,
            "kind": "greedy_accepted" if step.accepted else "greedy_rejected",
            "label": f"greedy {run_index + 1}.{step_index}",
            "F(C−K)": step.result.circuit_without.mean,
            "F(M−K)": step.result.model_without.mean,
            "incompleteness": step.result.incompleteness,
            "candidate_incompleteness": step.candidate_result.incompleteness,
        }
        greedy_rows.append(row)
        completeness_rows.append(row)
completeness_fig = px.scatter(
    completeness_rows,
    x="F(C−K)",
    y="F(M−K)",
    color="kind",
    hover_name="label",
    size="incompleteness",
    title="Completeness por classe, subconjunto aleatório e curva greedy",
)
all_values = [
    value
    for row in completeness_rows
    for value in (row["F(C−K)"], row["F(M−K)"])
]
line_min, line_max = min(all_values), max(all_values)
completeness_fig.add_trace(
    go.Scatter(
        x=[line_min, line_max],
        y=[line_min, line_max],
        mode="lines",
        line={"dash": "dash", "color": "gray"},
        name="F(C−K)=F(M−K)",
    )
)
completeness_fig.show()

greedy_fig = px.line(
    greedy_rows,
    x="step",
    y="incompleteness",
    color="run",
    symbol="accepted",
    markers=True,
    hover_data=["candidate_incompleteness", "label"],
    title="10 greedy runs; rejected steps are marked separately",
)
top5_rows = [
    {
        "step": item.step_index,
        "incompleteness": item.result.incompleteness,
        "label": f"top {rank}: run {item.run_index + 1}, step {item.step_index}",
    }
    for rank, item in enumerate(completeness.top_greedy, start=1)
]
greedy_fig.add_trace(
    go.Scatter(
        x=[row["step"] for row in top5_rows],
        y=[row["incompleteness"] for row in top5_rows],
        text=[row["label"] for row in top5_rows],
        mode="markers+text",
        marker={"size": 13, "symbol": "star", "color": "black"},
        textposition="top center",
        name="global top 5 accepted sets",
    )
)
greedy_fig.show()

minimality_rows = [
    {
        "node": node_name(item.node),
        "minimality_percent": 100 * item.normalized_effect,
        "ci95_low_percent": 100 * item.normalized_effect_ci95.low,
        "raw_effect": item.raw_effect,
        "passes_1_percent": item.passes_threshold,
        "ci_supports_1_percent": item.ci_supports_threshold,
    }
    for item in minimality
]
minimality_fig = px.bar(
    minimality_rows,
    x="node",
    y="minimality_percent",
    color="ci_supports_1_percent",
    hover_data=["raw_effect", "passes_1_percent", "ci95_low_percent"],
    title="Minimality por node nos contextos históricos",
    labels={"minimality_percent": "% de |F(M)|"},
)
minimality_fig.add_hline(y=1.0, line_dash="dash", annotation_text="threshold do paper: 1%")
minimality_fig.show()